# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset contains mass spectrometry-based measurements of protein abundance, modifications, and sequences in human extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR^2 dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.date_published if hasattr(metadata, 'date_published') else metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s to understand the structure of the dataset.

**Note:** All entity references use their `@id` as per the Croissant specification.

In [ ]:
# List all RecordSets and their Fields/Columns
print("Available RecordSets and their @id and label:")
for record_set in dataset.record_sets:
    print(f"  RecordSet name: {record_set.name}")
    print(f"    @id: {record_set.id}")    
    print("    Fields and Columns:")
    for field in record_set.fields:
        print(f"      Field name: {field.name}, @id: {field.id}")
    if hasattr(record_set, 'columns'):
        for column in record_set.columns:
            print(f"      Column name: {column.name}, @id: {column.id}")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis using their `@id`. Fields and columns are referenced by their `@id`.

In [ ]:
# Extract data from all RecordSets
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"> Columns for {record_set_id}: {df.columns.tolist()}")
    display(df.head(2))  # show just top two for a quick look

# Choose the first record set with non-empty DataFrame for further EDA
main_record_set_id = None
for k, v in dataframes.items():
    if len(v) > 0:
        main_record_set_id = k
        break
if main_record_set_id is None:
    raise RuntimeError("No populated record sets found!")
print(f"▶️ Using RecordSet `{main_record_set_id}` for analysis.")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Reference all attributes by their `@id`.

We'll demonstrate filtering and normalization for a numeric field, and grouping by a likely categorical field based on the available columns in the main record set.

In [ ]:
# Choose a numeric field and a group field by @id
candidate_numeric_fields = []
candidate_group_fields = []

df = dataframes[main_record_set_id]

for col in df.columns:
    # try to detect likely numeric (float or int) fields
    if pd.api.types.is_numeric_dtype(df[col]):
        candidate_numeric_fields.append(col)
    elif df[col].dtype == 'object' and col.lower().startswith(('group', 'sample', 'category', 'type', 'class', 'acc')):
        candidate_group_fields.append(col)

print(f"Numeric field candidates: {candidate_numeric_fields}")
print(f"Group field candidates: {candidate_group_fields}")

# Select first numeric field
if not candidate_numeric_fields:
    raise RuntimeError("Could not find numeric fields in main RecordSet DataFrame.")
numeric_field_id = candidate_numeric_fields[0]
print(f"Using numeric field for analysis: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
group_field_id = candidate_group_fields[0] if candidate_group_fields else None
if group_field_id is not None and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. We'll plot the numeric field's distribution and, if a group field is available, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group field if present
if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR^2 Croissant dataset and explored its structure and metadata using `mlcroissant`.
- Listed the available record sets, fields, and their `@id`s per FAIR standards.
- Extracted the main protein quantitative record set into a DataFrame, then performed exploratory analyses: filtering, normalization, and grouping.
- Visualized key data distributions.

**You can use these techniques to further analyze protein abundance and compare modifications across experimental conditions using the precise field and record set `@id`s.**